In [1]:
# ============================================================
# IMPORTS
# ============================================================

import itertools
import math
import numpy as np
import pandas as pd
import simpy

In [2]:
# ============================================================
# PARAMETRIC DISTRIBUTIONS
# ============================================================

class Exponential:
    def __init__(self, mean, random_seed=None):
        self.mean = float(mean)
        self.rng = np.random.default_rng(seed=random_seed)

    def sample(self, size=None):
        return self.rng.exponential(self.mean, size=size)


class Lognormal:
    def __init__(self, mean, sd, random_seed=None):
        mean = float(mean)
        sd = float(sd)

        sigma2 = math.log(1.0 + (sd ** 2) / (mean ** 2))
        self.mu = math.log(mean) - sigma2 / 2.0
        self.sigma = math.sqrt(sigma2)

        self.rng = np.random.default_rng(seed=random_seed)

    def sample(self, size=None):
        return self.rng.lognormal(self.mu, self.sigma, size=size)


class HyperExponential2:
    """
    Two-branch hyperexponential surrogate.

    This is used because the paper reports hyperexponential LOS
    for some streams, but not the exact phase parameters.
    """
    def __init__(self, mean, sd, random_seed=None):
        mean = float(mean)
        sd = float(sd)

        cv2 = (sd / mean) ** 2
        if cv2 <= 1.0:
            raise ValueError("HyperExponential2 requires sd > mean")

        self.p = 0.5 * (1.0 + math.sqrt((cv2 - 1.0) / (cv2 + 1.0)))
        self.lambda1 = 2.0 * self.p / mean
        self.lambda2 = 2.0 * (1.0 - self.p) / mean

        self.rng = np.random.default_rng(seed=random_seed)

    def sample(self, size=None):
        if size is None:
            if self.rng.random() < self.p:
                return self.rng.exponential(1.0 / self.lambda1)
            return self.rng.exponential(1.0 / self.lambda2)

        u = self.rng.random(size=size)
        x1 = self.rng.exponential(1.0 / self.lambda1, size=size)
        x2 = self.rng.exponential(1.0 / self.lambda2, size=size)
        return np.where(u < self.p, x1, x2)

In [3]:
# ============================================================
# GROUPED EMPIRICAL DISTRIBUTION
# ============================================================
# Expected CSV columns:
# - lower_bound
# - upper_bound
# - y
# ============================================================

class GroupedEmpirical:
    def __init__(self, csv_path, random_seed=None):
        self.df = pd.read_csv(csv_path)
        self.rng = np.random.default_rng(seed=random_seed)

        required = {"lower_bound", "upper_bound", "y"}
        missing = required.difference(self.df.columns)
        if missing:
            raise ValueError(f"Missing columns in empirical CSV: {sorted(missing)}")

        weights = self.df["y"].astype(float).values
        self.prob = weights / weights.sum()

        self.lower = self.df["lower_bound"].astype(float).values
        self.upper = self.df["upper_bound"].astype(float).values

    def sample(self, size=None):
        if size is None:
            idx = self.rng.choice(len(self.prob), p=self.prob)
            return self.rng.uniform(self.lower[idx], self.upper[idx])

        out = []
        for _ in range(size):
            idx = self.rng.choice(len(self.prob), p=self.prob)
            out.append(self.rng.uniform(self.lower[idx], self.upper[idx]))
        return np.array(out)

In [4]:
# ============================================================
# IMPROVED ELECTIVE SCHEDULER
# ============================================================
# This keeps the empirical inter-arrival information, but uses it
# inside a weekly elective schedule rather than as a pure renewal
# process running forever.
#
# Logic:
# - build one week at a time
# - accumulate empirical inter-arrival gaps within the working part
#   of the week
# - optional weekday weighting can spread the weekly schedule
# ============================================================

class EmpiricalElectiveScheduler:
    def __init__(
        self,
        csv_path,
        random_seed=None,
        max_week_hours=5 * 24,
        week_length_hours=7 * 24,
        start_offset=0.0,
        weekday_weights=None,
    ):
        self.empirical_iat = GroupedEmpirical(csv_path, random_seed=random_seed)
        self.rng = np.random.default_rng(seed=random_seed)
        self.max_week_hours = float(max_week_hours)
        self.week_length_hours = float(week_length_hours)
        self.start_offset = float(start_offset)

        if weekday_weights is None:
            weekday_weights = [0.19, 0.21, 0.21, 0.20, 0.19]

        self.weekday_weights = np.asarray(weekday_weights, dtype=float)
        self.weekday_weights = self.weekday_weights / self.weekday_weights.sum()

    def sample_week_schedule(self):
        """
        Build one week's elective arrivals.

        Returns
        -------
        list of float
            Arrival offsets in hours from the start of the week.
        """
        arrivals = []
        t = self.start_offset

        while t < self.max_week_hours:
            iat = self.empirical_iat.sample()
            t += iat
            if t < self.max_week_hours:
                arrivals.append(t)

        # Optional redistribution across weekdays while preserving
        # within-day timing information as much as possible.
        #
        # This is a mild smoothing step so that all arrivals do not
        # accumulate disproportionately early in the week.
        redistributed = []
        for arrival in arrivals:
            hour_within_day = arrival % 24.0
            chosen_day = self.rng.choice(5, p=self.weekday_weights)
            redistributed.append(chosen_day * 24.0 + hour_within_day)

        redistributed.sort()
        return redistributed

In [5]:
# ============================================================
# SCENARIO CLASS
# ============================================================
# This version is:
# - self-contained
# - robust
# - clear about what is empirical vs parametric
#
# Elective arrivals:
#   empirical weekly scheduler
# Elective LOS:
#   parametric surrogate (still hybrid, but improved on the arrival side)
# ============================================================

class Scenario:
    def __init__(self, beds=24, random_seed=42, elective_iat_csv="data(in).csv"):
        self.beds = int(beds)
        self.random_seed = int(random_seed)
        self.elective_iat_csv = elective_iat_csv

        # Global arrival scaling for non-elective streams
        arrival_scale = 1.00

        # Non-elective arrivals remain parametric
        self.arrival_dist = {
            "ae": Exponential(22.72 * arrival_scale, random_seed + 1),
            "ward": Exponential(26.00 * arrival_scale, random_seed + 2),
            "emergency": Exponential(37.00 * arrival_scale, random_seed + 3),
            "other": Exponential(47.20 * arrival_scale, random_seed + 4),
            "xray": Exponential(575.00 * arrival_scale, random_seed + 5),
        }

        # Elective arrivals use empirical weekly scheduling
        self.elective_scheduler = EmpiricalElectiveScheduler(
            csv_path=elective_iat_csv,
            random_seed=random_seed + 6,
            max_week_hours=5 * 24,
            week_length_hours=7 * 24,
            start_offset=0.0,
            weekday_weights=[0.19, 0.21, 0.21, 0.20, 0.19],
        )

        # LOS distributions
        self.los_dist = {
            "ae": HyperExponential2(109.0, 218.0, random_seed + 11),
            "ward": HyperExponential2(152.0, 238.0, random_seed + 12),
            "emergency": HyperExponential2(119.0, 187.0, random_seed + 13),
            "other": Lognormal(182.0, 345.0, random_seed + 14),
            "xray": Lognormal(79.0, 93.0, random_seed + 15),

            # Elective LOS is still parametric here because LOS data
            # were not provided as empirical observations.
            "elective": Lognormal(31.0, 42.0, random_seed + 16),
        }

        # Cleaning / turnaround time after discharge
        self.cleaning_time = 5.0

        # Simulation design
        self.warm_up = 30 * 24
        self.run_length = 365 * 24

        # To be attached after environment creation
        self.beds_resource = None
        self.auditor = None

In [6]:
# ============================================================
# AUDITOR
# ============================================================
# Tracks:
# - time-weighted occupancy
# - cancellations
# - admissions
# - unplanned waiting
# ============================================================

class Auditor:
    def __init__(self, warm_up, run_length):
        self.warm_up = float(warm_up)
        self.run_length = float(run_length)
        self.end_time = self.warm_up + self.run_length

        self.metrics = {
            "occupied_bed_area": 0.0,
            "cancellations": 0,
            "admissions": 0,
            "elective_arrivals": 0,
            "elective_admitted": 0,
            "unplanned_admitted": 0,
            "unplanned_wait_sum": 0.0,
            "unplanned_wait_count": 0,
            "unplanned_wait_max": 0.0,
        }

        self.last_time = 0.0
        self.last_occupied = 0

    def update_occupied(self, now, occupied):
        start = max(self.last_time, self.warm_up)
        stop = min(now, self.end_time)

        if stop > start:
            self.metrics["occupied_bed_area"] += self.last_occupied * (stop - start)

        self.last_time = now
        self.last_occupied = occupied

    def record_cancellation(self, now):
        if now >= self.warm_up:
            self.metrics["cancellations"] += 1

    def record_elective_arrival(self, now):
        if now >= self.warm_up:
            self.metrics["elective_arrivals"] += 1

    def record_admission(self, source, now):
        if now >= self.warm_up:
            self.metrics["admissions"] += 1
            if source == "elective":
                self.metrics["elective_admitted"] += 1
            else:
                self.metrics["unplanned_admitted"] += 1

    def record_unplanned_wait(self, wait, now):
        if now >= self.warm_up:
            self.metrics["unplanned_wait_sum"] += wait
            self.metrics["unplanned_wait_count"] += 1
            if wait > self.metrics["unplanned_wait_max"]:
                self.metrics["unplanned_wait_max"] = wait

    def finalize(self):
        self.update_occupied(self.end_time, self.last_occupied)

    def summary(self, beds):
        mean_occupied = self.metrics["occupied_bed_area"] / self.run_length
        occupancy_pct = 100.0 * mean_occupied / beds

        if self.metrics["unplanned_wait_count"] > 0:
            avg_unplanned_wait = (
                self.metrics["unplanned_wait_sum"] / self.metrics["unplanned_wait_count"]
            )
        else:
            avg_unplanned_wait = 0.0

        return {
            "mean_occupied": mean_occupied,
            "occupancy_pct": occupancy_pct,
            "cancellations": self.metrics["cancellations"],
            "admissions": self.metrics["admissions"],
            "elective_arrivals": self.metrics["elective_arrivals"],
            "elective_admitted": self.metrics["elective_admitted"],
            "unplanned_admitted": self.metrics["unplanned_admitted"],
            "avg_unplanned_wait": avg_unplanned_wait,
            "max_unplanned_wait": self.metrics["unplanned_wait_max"],
        }

In [7]:
# ============================================================
# PATIENT ENTITY
# ============================================================
# Rules:
# - elective: cancel if no bed immediately available
# - unplanned: wait until a bed becomes available
# ============================================================

class Patient:
    def __init__(self, source, env, args):
        self.source = source
        self.env = env
        self.args = args

    @property
    def beds(self):
        return self.args.beds_resource

    @property
    def auditor(self):
        return self.args.auditor

    def service(self):
        arrival_time = self.env.now

        if self.source == "elective":
            self.auditor.record_elective_arrival(self.env.now)

            # Elective admissions are lost if no bed is available immediately
            if self.beds.level < 1:
                self.auditor.record_cancellation(self.env.now)
                return

            yield self.beds.get(1)
            self.auditor.update_occupied(self.env.now, self.args.beds - self.beds.level)
            self.auditor.record_admission(self.source, self.env.now)

        else:
            # Unplanned patients wait until a bed is available
            yield self.beds.get(1)

            wait_time = self.env.now - arrival_time
            self.auditor.record_unplanned_wait(wait_time, self.env.now)

            self.auditor.update_occupied(self.env.now, self.args.beds - self.beds.level)
            self.auditor.record_admission(self.source, self.env.now)

        # Stay in bed according to source-specific LOS
        los = self.args.los_dist[self.source].sample()
        yield self.env.timeout(los)

        # Occupancy drops immediately at discharge
        self.auditor.update_occupied(self.env.now, self.args.beds - self.beds.level - 1)

        # Bed unavailable during cleaning / turnaround
        yield self.env.timeout(self.args.cleaning_time)

        # Return the bed to available capacity
        yield self.beds.put(1)

In [8]:
# ============================================================
# MODEL ENGINE
# ============================================================
# Non-elective arrivals:
#   renewal processes from exponential distributions
#
# Elective arrivals:
#   week-by-week schedules built from empirical inter-arrival data
# ============================================================

class CCUBedModel:
    def __init__(self, env, args):
        self.env = env
        self.args = args
        self.patient_counter = itertools.count(start=1)

    def non_elective_arrivals(self, source):
        while True:
            iat = self.args.arrival_dist[source].sample()
            yield self.env.timeout(iat)

            _ = next(self.patient_counter)
            patient = Patient(source, self.env, self.args)
            self.env.process(patient.service())

    def elective_arrivals(self):
        while True:
            weekly_offsets = self.args.elective_scheduler.sample_week_schedule()

            for offset in weekly_offsets:
                self.env.process(self._elective_after_delay(offset))

            yield self.env.timeout(7 * 24)

    def _elective_after_delay(self, delay_hours):
        yield self.env.timeout(delay_hours)

        _ = next(self.patient_counter)
        patient = Patient("elective", self.env, self.args)
        self.env.process(patient.service())

In [9]:
# ============================================================
# SINGLE RUN
# ============================================================

def single_run(beds=24, seed=42, csv="data(in).csv"):
    env = simpy.Environment()
    args = Scenario(beds=beds, random_seed=seed, elective_iat_csv=csv)

    args.auditor = Auditor(args.warm_up, args.run_length)
    args.beds_resource = simpy.Container(env, init=beds, capacity=beds)

    model = CCUBedModel(env, args)

    # Start non-elective streams
    for src in ["ae", "ward", "emergency", "other", "xray"]:
        env.process(model.non_elective_arrivals(src))

    # Start weekly elective scheduling
    env.process(model.elective_arrivals())

    env.run(until=args.warm_up + args.run_length)
    args.auditor.finalize()

    results = args.auditor.summary(beds)
    results["beds"] = beds
    results["seed"] = seed
    return results

In [10]:
# ============================================================
# REPLICATION HELPERS
# ============================================================

def run_replications(beds=24, n_reps=20, base_seed=1000, csv="data(in).csv"):
    rows = []
    for rep in range(n_reps):
        rows.append(single_run(beds=beds, seed=base_seed + rep, csv=csv))
    return pd.DataFrame(rows)


def summary_from_replications(df):
    return pd.Series({
        "beds": int(df["beds"].iloc[0]),
        "mean_occupied": df["mean_occupied"].mean(),
        "occupancy_pct": df["occupancy_pct"].mean(),
        "cancellations": df["cancellations"].mean(),
        "admissions": df["admissions"].mean(),
        "elective_arrivals": df["elective_arrivals"].mean(),
        "elective_admitted": df["elective_admitted"].mean(),
        "unplanned_admitted": df["unplanned_admitted"].mean(),
        "avg_unplanned_wait": df["avg_unplanned_wait"].mean(),
        "max_unplanned_wait": df["max_unplanned_wait"].mean(),
    })


def run_bed_scenarios(bed_range=range(22, 30), n_reps=20, base_seed=1000, csv="data(in).csv"):
    summaries = []
    for beds in bed_range:
        df = run_replications(
            beds=beds,
            n_reps=n_reps,
            base_seed=base_seed + beds * 100,
            csv=csv,
        )
        summaries.append(summary_from_replications(df))
    return pd.DataFrame(summaries)

In [11]:
# ============================================================
# PAPER BENCHMARK TABLE
# ============================================================

def paper_table_2_targets():
    return pd.DataFrame({
        "beds": [22, 23, 24, 25, 26, 27, 28, 29],
        "paper_mean_occupied": [19.15, 19.40, 19.76, 20.05, 20.18, 20.20, 20.20, 20.20],
        "paper_occupancy_pct": [87, 84, 82, 80, 78, 75, 72, 70],
        "paper_cancellations": [146, 101, 57, 20, 3, 0, 0, 0],
    })


def compare_to_paper(n_reps=20, csv="data(in).csv"):
    sim_df = run_bed_scenarios(n_reps=n_reps, csv=csv)
    target_df = paper_table_2_targets()

    out = sim_df.merge(target_df, on="beds", how="left")
    out["err_mean_occupied"] = out["mean_occupied"] - out["paper_mean_occupied"]
    out["err_occupancy_pct"] = out["occupancy_pct"] - out["paper_occupancy_pct"]
    out["err_cancellations"] = out["cancellations"] - out["paper_cancellations"]
    return out

In [12]:
# ============================================================
# QUICK SANITY CHECKS
# ============================================================

print("24-bed run")
print(pd.Series(single_run(beds=24, seed=42, csv="data(in).csv")))
print()

print("Extreme capacity check")
for b in [1, 10, 24, 50]:
    res = single_run(beds=b, seed=42, csv="data(in).csv")
    print(f"Beds={b:>2} | Occ={res['occupancy_pct']:.2f}% | Cancels={res['cancellations']:.0f} | Avg unplanned wait={res['avg_unplanned_wait']:.2f}")

24-bed run
mean_occupied           20.078988
occupancy_pct           83.662452
cancellations           59.000000
admissions            1450.000000
elective_arrivals      297.000000
elective_admitted      238.000000
unplanned_admitted    1212.000000
avg_unplanned_wait       3.643718
max_unplanned_wait      70.535834
beds                    24.000000
seed                    42.000000
dtype: float64

Extreme capacity check
Beds= 1 | Occ=97.83% | Cancels=297 | Avg unplanned wait=5838.68
Beds=10 | Occ=97.45% | Cancels=297 | Avg unplanned wait=2424.01
Beds=24 | Occ=83.66% | Cancels=59 | Avg unplanned wait=3.64
Beds=50 | Occ=40.84% | Cancels=0 | Avg unplanned wait=0.00


In [13]:
results_clean = run_bed_scenarios(
    bed_range=range(22, 30),
    n_reps=20,
    csv="data(in).csv"
)
cols = [
    "beds",
    "mean_occupied",
    "occupancy_pct",
    "cancellations",
    "admissions"
]

print(results_clean[cols].round(2).to_string(index=False))

 beds  mean_occupied  occupancy_pct  cancellations  admissions
 22.0          19.18          87.18         131.75     1332.10
 23.0          19.29          83.87          98.05     1381.25
 24.0          19.29          80.36          68.20     1402.65
 25.0          19.63          78.52          57.60     1427.75
 26.0          19.80          76.15          41.05     1419.00
 27.0          19.62          72.68          21.65     1425.35
 28.0          19.68          70.30          14.40     1464.10
 29.0          19.47          67.14           7.15     1457.80
